# Van Hateren：从缓存重画完整比较图

直接 **Run All**，不重跑 MC。三列是固定正则、DE generalization 选择、distributional accentuation oracle；四行是误差、R²、slope、正则。

数据：10,000 pixels，disk teacher，n=1000，100 次自然图像 MC。不是 Gaussian-design MC。颜色区分 gen / acc，线型区分估计方法。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'scripts/pixel_cv_stationary.py').exists())
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'notebooks/outputs/pixel_ridge/vanhateren_selection_estimation'
OUT = DATA / 'notebook_reproduction'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42,
                     'font.family': 'DejaVu Sans', 'mathtext.fontset': 'dejavusans'})
print('Cache:', DATA)
print('New outputs:', OUT)


## 1. 读取缓存，检查范围

MC 点是 arithmetic mean，误差棒是 ±2 SE；阴影是 trial 10–90% 范围，不是均值置信区间。distributional DE 是 moment-matched diagonal Gaussian surrogate，并非完整的 distributional RMT 定理。

In [ ]:
table = pd.read_csv(DATA / 'comparison_extended.csv')
cv = pd.read_csv(DATA / 'sklearn_cv_summary.csv')
interior = pd.read_csv(DATA / 'stationary_cv.csv')
extrapolated = pd.read_csv(DATA / 'stationary_cv_extrapolated.csv')
display(table.head())
print('sigma range:', table.sigma.min(), table.sigma.max())
print('columns:', table.columns.tolist())
display(interior[['sigma', 'status', 'kappa', 'lam', 'z_acc']])


## 2. 选择显示内容

sklearn RidgeCV 仅叠加在中间列，因为它也是 prediction-based selection。

`EXTRAPOLATE_STATIONARY=True`：低噪声在 κ₀ 直接代入论文公式，以空心方块标注。它不是合法的内部驻点预测。False 则保持空白。

论文驻点使用 n=1000；缓存的 DE-gen selection 使用 n−1 的 LOOCV proxy 后在 n 上评估，所以两条线不必完全相同。

In [ ]:
SHOW_SKLEARN_CV = True
SHOW_STATIONARY = True
EXTRAPOLATE_STATIONARY = True

from scripts.extend_vanhateren_evaluations import plot
stationary = (extrapolated if EXTRAPOLATE_STATIONARY else interior) if SHOW_STATIONARY else None
fig = plot(table, sklearn_cv=cv if SHOW_SKLEARN_CV else None,
           show_sklearn_cv=SHOW_SKLEARN_CV, stationary=stationary)
plt.show()


## 3. 保存 PNG + vector-font PDF

只写入 notebook_reproduction，不覆盖原始缓存或原图。可以在上一 cell 的 fig.axes 上修改坐标范围和标题后重新保存。

In [ ]:
tag = f'comparison_cv{int(SHOW_SKLEARN_CV)}_stationary{int(SHOW_STATIONARY)}_extrapolate{int(EXTRAPOLATE_STATIONARY)}'
for ext in ('png', 'pdf'):
    path = OUT / f'{tag}.{ext}'
    fig.savefig(path, dpi=180, bbox_inches='tight')
    print(path)


## 指标定义与限制

这里 E_gen、E_acc 的表格列已除以 S。定义 slope_acc=N/D，E_acc/S=(1−N/D)²，而 R²_acc=1−(D/N−1)²；不能用 1−E_acc/S 替代 R²_acc。MC 的非线性指标先按 trial 计算再平均。

Gaussian surrogate 的 R²_acc 涉及 1/N，有限抽样结果可能受极端值影响，甚至对应的总体期望不有限。固定和 oracle 列的 MC 使用该列确定性选择的正则；sklearn 曲线才是每个 trial 自己选正则。

本 notebook 复用已经完成的 MC / distributional draws。第二个 notebook 独立重算解析驻点和边界比较。